# ClearPath Database Build — process validation

**date**: 2026-06-02/03  
**scope**: Manhattan only

This notebook contains the complete process:
1. Data source file validation
2. Schema structure validation
3. Data volume comparison (Manhattan only)
4. Data quality check
5. ETL data import
6. Import result validation
7. Schema update (API interface alignment)

In [ ]:
import csv, json, re, hashlib, datetime, os
from decimal import Decimal
from pathlib import Path
import pymysql

def discover_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    return next((candidate for candidate in candidates if (candidate / 'Data+ML').is_dir()), Path.cwd())

PROJECT_ROOT = Path(
    os.environ.get('CLEARPATH_PROJECT_ROOT', discover_project_root())
).expanduser().resolve()
DATA_ROOT = Path(
    os.environ.get('CLEARPATH_DATA_ROOT', PROJECT_ROOT.parent / 'data_source')
).expanduser().resolve()
MANIFEST_PATH = PROJECT_ROOT / 'Data+ML/test/6.2-6.5_DB/clearpath_sources.json'
SCHEMA_PATH = PROJECT_ROOT / 'Data+ML/test/6.2-6.5_DB/001_clearpath_schema.sql'
MYSQL_CONFIG = {
    'host': os.environ.get('CLEARPATH_DB_HOST', '127.0.0.1'),
    'port': int(os.environ.get('CLEARPATH_DB_PORT', '3306')),
    'user': os.environ.get('CLEARPATH_DB_USER', 'clearpath_app'),
    'password': os.environ.get('CLEARPATH_DB_PASSWORD', 'clearpath_app'),
    'database': os.environ.get('CLEARPATH_DB_NAME', 'clearpath'),
    'charset': 'utf8mb4',
}
MANHATTAN_BBOX = {'lat_min': 40.700, 'lat_max': 40.880, 'lng_min': -74.020, 'lng_max': -73.9}
NYC_BBOX = {'lat_min': 40.490, 'lat_max': 40.920, 'lng_min': -74.260, 'lng_max': -73.700}
COUNTY_BOROUGH = {'New York': 'Manhattan', 'Kings': 'Brooklyn', 'Queens': 'Queens', 'Bronx': 'Bronx', 'Richmond': 'Staten Island'}
OSM_CATEGORY_MAP = {'hospital': 'hospital', 'clinic': 'clinic', 'doctors': 'clinic', 'pharmacy': 'pharmacy', 'dentist': 'dentist', 'laboratory': 'laboratory'}

def is_manhattan(lat, lng):
    return MANHATTAN_BBOX['lat_min'] <= float(lat) <= MANHATTAN_BBOX['lat_max'] and MANHATTAN_BBOX['lng_min'] <= float(lng) <= MANHATTAN_BBOX['lng_max']

def gps_to_district(lat, lng):
    """Manhattan GPS coordinates → administrative district name.
    
    Districts: uptown, midtown_east, midtown_west, downtown
    Based on Pipeline requirements for zoned queries and ML aggregation.
    """
    lat, lng = float(lat), float(lng)
    if lat > 40.800:
        return 'uptown'
    if lat > 40.750:
        return 'midtown_east' if lng > -73.975 else 'midtown_west'
    return 'downtown'

def source_hash(*parts):
    return hashlib.sha256('|'.join(str(p) for p in parts if p).encode()).hexdigest()[:36]

def gen_vid(source, sid):
    return source_hash(source, sid)

def get_conn():
    return pymysql.connect(**MYSQL_CONFIG)

def safe_int(v):
    try:
        return int(float(str(v).strip())) if v and str(v).strip() else None
    except (ValueError, TypeError, OverflowError):
        return None

def safe_dec(v):
    try:
        return Decimal(str(v).strip()) if v and str(v).strip() else None
    except (ValueError, TypeError, ArithmeticError):
        return None

def column_exists(conn, table, column):
    with conn.cursor() as cur:
        cur.execute(
            "SELECT COUNT(*) FROM information_schema.COLUMNS "
            "WHERE TABLE_SCHEMA = %s AND TABLE_NAME = %s AND COLUMN_NAME = %s",
            (MYSQL_CONFIG['database'], table, column),
        )
        return cur.fetchone()[0] > 0

def table_exists(conn, table):
    with conn.cursor() as cur:
        cur.execute(
            "SELECT COUNT(*) FROM information_schema.TABLES "
            "WHERE TABLE_SCHEMA = %s AND TABLE_NAME = %s",
            (MYSQL_CONFIG['database'], table),
        )
        return cur.fetchone()[0] > 0

def log_etl_error(source, record_id, error):
    print(f'  [{source}] record={record_id!r} failed: {type(error).__name__}: {error}')

def etl_execute(conn, statements, *, source, record_id):
    if isinstance(statements, tuple) and len(statements) == 2 and isinstance(statements[0], str):
        statements = [statements]
    try:
        with conn.cursor() as cur:
            for sql, params in statements:
                cur.execute(sql, params)
        conn.commit()
        return True
    except pymysql.MySQLError as error:
        conn.rollback()
        log_etl_error(source, record_id, error)
        return False

def etl_executemany(conn, sql, params, *, source, record_id):
    try:
        with conn.cursor() as cur:
            cur.executemany(sql, params)
        conn.commit()
        return len(params), 0
    except pymysql.MySQLError as error:
        conn.rollback()
        log_etl_error(source, record_id, error)
        return 0, len(params)

print('Tools loaded')

---
## Part 1: Data Source Validation

In [ ]:
with open(MANIFEST_PATH, encoding='utf-8') as f:
    manifest = json.load(f)
local_sources = [s for s in manifest['included_sources'] if s.get('local_file')]
print(f'Manifest: {len(manifest["included_sources"])} sources, {len(local_sources)} local files')
for source in local_sources:
    fpath = DATA_ROOT / source['local_file']
    if not fpath.exists():
        print(f'MISSING: {source["local_file"]}'); continue
    if fpath.suffix == '.csv':
        with open(fpath, encoding='utf-8-sig') as f:
            count = sum(1 for _ in csv.reader(f)) - 1
        print(f'OK {source["local_file"]}: {count} rows')
    elif fpath.suffix in ('.json', '.geojson'):
        with open(fpath, encoding='utf-8') as f:
            count = len(json.load(f).get('features', []))
        print(f'OK {source["local_file"]}: {count} features')

---
## Part 2: Schema Validation

In [ ]:
schema_sql = SCHEMA_PATH.read_text(encoding='utf-8')
# Support both "CREATE TABLE" and "CREATE TABLE IF NOT EXISTS" formats
actual_tables = set(re.findall(r'CREATE TABLE\s+(?:IF NOT EXISTS\s+)?`?([a-z_]+)`?', schema_sql))
expected_tables = {'venues', 'venue_source_links', 'restroom_profiles', 'healthcare_profiles', 'emergency_assets', 'pedestrian_ramps', 'user_reports', 'report_confirmations', 'busyness_scores', 'external_context_cache', 'venue_accessibility', 'venue_language', 'venue_warnings', 'users', 'user_favorite_venues', 'notification_preferences', 'report_categories', 'busyness_forecasts', 'venue_embeddings'}
print(f'Schema tables: {len(actual_tables)}, expected: {len(expected_tables)}, match: {actual_tables == expected_tables}')
excluded = [t for t in ['poi_accessibility', 'hrsa', 'citymd', 'google_places', 'mta', 'taxi', 'traffic_volume', 'language_access', 'lep'] if t in schema_sql.lower()]
print(f'Excluded terms: {excluded if excluded else "none"}')

---
## Part 3: Data Volume Comparison (Manhattan)

### Filter Method

Each data source uses a different method to limit records to Manhattan:

| Data Source | Filter Method | Code | Notes |
|-------------|:-------------:|------|-------|
| Public Restrooms | GPS bounding box | `is_manhattan(lat, lng)` | lat 40.700~40.880, lng -74.020~-73.907 |
| Parks Toilets | Borough field | `r.get('Borough') == 'manhattan'` | No coordinates in source; Borough name only |
| OSM Healthcare | GPS bounding box | `is_manhattan(lat, lng)` | Extracted from GeoJSON `geometry.coordinates` |
| NYS Health Facility | County field | `r.get('Facility County') == 'New York'` | New York County = Manhattan |
| AED Inventory | Borough field | `r.get('Borough') == 'manhattan'` | Borough column provided by source |
| Pedestrian Ramps | Borough code | `r.get('Borough') == '1'` | Manhattan Borough code is `'1'` |

**Note**: In Part 5 ETL, `etl_nys` must use the same filter as Part 3 (`County == 'New York'` only), to avoid importing data from other boroughs.

In [ ]:
with open(DATA_ROOT / 'Public_Restrooms_20260526.csv', encoding='utf-8-sig') as f: restrooms = list(csv.DictReader(f))
with open(DATA_ROOT / 'Directory_Of_Toilets_In_Public_Parks_20260526.csv', encoding='utf-8-sig') as f: parks = list(csv.DictReader(f))
with open(DATA_ROOT / 'POI_healtcare.geojson', encoding='utf-8') as f: osm_features = json.load(f).get('features', [])
with open(DATA_ROOT / 'Health_Facility_General_Information_20260526.csv', encoding='utf-8-sig') as f: nys = list(csv.DictReader(f))
with open(DATA_ROOT / 'New_York_City_Automated_External_Defibrillator_(AED)_Inventory_20260526.csv', encoding='utf-8-sig') as f: aed = list(csv.DictReader(f))
with open(DATA_ROOT / 'Pedestrian_Ramp_Locations_20260526.csv', encoding='utf-8-sig') as f: ramps = list(csv.DictReader(f))
restrooms_m = [r for r in restrooms if is_manhattan(float(r.get('Latitude', 0) or 0), float(r.get('Longitude', 0) or 0))]
parks_m = [r for r in parks if (r.get('Borough') or '').strip().lower() == 'manhattan']
osm_m = [f for f in osm_features if len(f.get('geometry', {}).get('coordinates', [])) >= 2 and is_manhattan(f['geometry']['coordinates'][1], f['geometry']['coordinates'][0])]
nys_m = [r for r in nys if (r.get('Facility County') or '').strip() == 'New York']
aed_m = [r for r in aed if (r.get('Borough') or '').strip().lower() == 'manhattan']
ramps_m = [r for r in ramps if (r.get('Borough') or '').strip() == '1']
EXPECTED = {'NYC Public Restrooms': 358, 'Parks Toilets': 129, 'OSM Healthcare': 900, 'NYS Health Facility': 454, 'AED Inventory': 3393, 'Pedestrian Ramps': 23625}
actual = {'NYC Public Restrooms': len(restrooms_m), 'Parks Toilets': len(parks_m), 'OSM Healthcare': len(osm_m), 'NYS Health Facility': len(nys_m), 'AED Inventory': len(aed_m), 'Pedestrian Ramps': len(ramps_m)}
print(f'{"Source":<25} {"Expected":>8} {"Actual":>8} {"Diff":>8} {"Diff%":>8}')
print('-' * 60)
for source, exp in EXPECTED.items():
    act = actual.get(source, 0)
    diff, pct = act - exp, ((act - exp) / exp * 100) if exp else 0
    print(f'{"OK" if abs(pct) < 20 else "WARN"} {source:<23} {exp:>8} {act:>8} {diff:>8} {pct:>7.1f}%')

---
## Part 4: Data Quality Check

**Scope**: Record-level field completeness analysis for all 6 data sources

**Tables Checked**:

| Source Table | Check Properties | Reason |
|--------------|------------------|--------|
| `restroom_profiles` | Coords (lat/lng), Name | GPS needed for map display |
| `healthcare_profiles` | Coords (lat/lng), Name | GPS needed for map display |
| `emergency_assets` | Coords (lat/lng), Name | GPS needed for map display |
| `pedestrian_ramps` | Coords (POINT parse), RampID | Geometry needed for routing |
| `venue_language` | GPS coverage, Language tags | GPS match needed for venue linkage |

**Output**: Field completeness % per source → identify sources with quality issues → pre-ETL decision

In [ ]:
# ============================================================
# Part 4: Data Quality Check — Record-level Analysis
# ============================================================

print("=" * 60)
print("Step 1: Field Completeness Analysis (all 6 data sources)")
print("=" * 60)

# 1. NYC Restrooms
restrooms_coords_ok = sum(1 for r in restrooms_m if r.get('Latitude', '').strip() and r.get('Longitude', '').strip())
restrooms_name_ok = sum(1 for r in restrooms_m if r.get('Facility Name', '').strip())
print(f"\nNYC Public Restrooms ({len(restrooms_m)} records):")
print(f"  Coords:   {restrooms_coords_ok}/{len(restrooms_m)} ({restrooms_coords_ok/len(restrooms_m)*100:.0f}%)")
print(f"  Name:     {restrooms_name_ok}/{len(restrooms_m)} ({restrooms_name_ok/len(restrooms_m)*100:.0f}%)")

# 2. Parks Toilets
parks_name_ok = sum(1 for r in parks_m if r.get('Name', '').strip())
print(f"\nParks Toilets ({len(parks_m)} records):")
print(f"  Coords:   0/{len(parks_m)} (0%) — ⚠️ NO coordinate field in source CSV")
print(f"  Name:     {parks_name_ok}/{len(parks_m)} ({parks_name_ok/len(parks_m)*100:.0f}%)")

# 3. OSM Healthcare
osm_coords_ok = sum(1 for f in osm_m if len(f.get('geometry', {}).get('coordinates', [])) >= 2)
osm_name_ok = sum(1 for f in osm_m if f.get('properties', {}).get('name', '').strip())
osm_no_name = [f for f in osm_m if not f.get('properties', {}).get('name', '').strip()]
print(f"\nOSM Healthcare ({len(osm_m)} records):")
print(f"  Coords:   {osm_coords_ok}/{len(osm_m)} ({osm_coords_ok/len(osm_m)*100:.0f}%)")
print(f"  Name:     {osm_name_ok}/{len(osm_m)} ({osm_name_ok/len(osm_m)*100:.0f}%) — ⚠️ {len(osm_no_name)} records missing name")

# 4. NYS Health
nys_coords_ok = sum(1 for r in nys_m if r.get('Facility Latitude', '').strip() and r.get('Facility Longitude', '').strip())
nys_no_coords = [r for r in nys_m if not r.get('Facility Latitude', '').strip() or not r.get('Facility Longitude', '').strip()]
nys_name_ok = sum(1 for r in nys_m if r.get('Facility Name', '').strip())
print(f"\nNYS Health Facility ({len(nys_m)} records):")
print(f"  Coords:   {nys_coords_ok}/{len(nys_m)} ({nys_coords_ok/len(nys_m)*100:.0f}%) — ⚠️ {len(nys_no_coords)} records missing coords")
print(f"  Name:     {nys_name_ok}/{len(nys_m)} ({nys_name_ok/len(nys_m)*100:.0f}%)")

# 5. AED Inventory
aed_coords_ok = sum(1 for r in aed_m if r.get('Latitude', '').strip() and r.get('Longitude', '').strip())
aed_name_ok = sum(1 for r in aed_m if r.get('Entity_Name', '').strip())
print(f"\nAED Inventory ({len(aed_m)} records):")
print(f"  Coords:   {aed_coords_ok}/{len(aed_m)} ({aed_coords_ok/len(aed_m)*100:.0f}%)")
print(f"  Name:     {aed_name_ok}/{len(aed_m)} ({aed_name_ok/len(aed_m)*100:.0f}%)")

# 6. Pedestrian Ramps
ramps_coords_ok = sum(1 for r in ramps_m if re.match(r'POINT\s*\(', r.get('the_geom', '')))
ramps_id_ok = sum(1 for r in ramps_m if r.get('RampID', '').strip())
print(f"\nPedestrian Ramps ({len(ramps_m)} records):")
print(f"  Coords:   {ramps_coords_ok}/{len(ramps_m)} ({ramps_coords_ok/len(ramps_m)*100:.0f}%) — via regex parse")
print(f"  RampID:   {ramps_id_ok}/{len(ramps_m)} ({ramps_id_ok/len(ramps_m)*100:.0f}%)")

# ============================================================
# Step 2: Identify sources with quality issues
# ============================================================
print("\n" + "=" * 60)
print("Step 2: Identify Sources with Quality Issues")
print("=" * 60)

issues = []
if len(nys_no_coords) > 0:
    issues.append(f"NYS Health: {len(nys_no_coords)} records missing coordinates")
issues.append(f"Parks Toilets: ALL {len(parks_m)} records missing coordinates (no field in source)")
if len(osm_no_name) > 0:
    issues.append(f"OSM Healthcare: {len(osm_no_name)} records missing name")

for i, issue in enumerate(issues, 1):
    print(f"  {i}. {issue}")

# ============================================================
# Step 3: Venue Language (LASS) — pre-ETL check
# ============================================================
print("\n" + "=" * 60)
print("Step 3: Venue Language (LASS) Pre-ETL Check")
print("=" * 60)

LASS_CHECK_PATH = DATA_ROOT / 'Language_Access_Secret_Shopper_(LASS)_Ratings_20260526.csv'
try:
    with open(LASS_CHECK_PATH, encoding='utf-8-sig') as f:
        lass_check = list(csv.DictReader(f))
    lass_m = [r for r in lass_check if (r.get('Borough') or '').strip().lower() == 'manhattan']
    lass_gps = [r for r in lass_m if r.get('Latitude', '').strip() and r.get('Longitude', '').strip()]
    lass_signs = [r for r in lass_m if r.get('Languages in which the facility has translated signs relating to service being provided', '').strip() not in ('', 'None', 'N/A')]
    lass_docs = [r for r in lass_m if r.get('Languages in which the facility has translated documents', '').strip() not in ('', 'None', 'N/A')]
    print(f"  Total records:      {len(lass_check)}")
    print(f"  Manhattan:          {len(lass_m)}")
    print(f"  With GPS:           {len(lass_gps)} ({len(lass_gps)/len(lass_m)*100:.1f}%)")
    print(f"  With signs langs:   {len(lass_signs)} ({len(lass_signs)/len(lass_m)*100:.1f}%)")
    print(f"  With docs langs:    {len(lass_docs)} ({len(lass_docs)/len(lass_m)*100:.1f}%)")
    print(f"  Estimated matches:  ~61 venues (GPS <100m threshold)")
except Exception as e:
    print(f"  LASS check failed: {e}")

---
## Part 5: ETL Data Import

**ETL Pipeline Flow**:
1. Schema Rebuild → 2. Dedup Preprocessing → 3. Restrooms → 4. Healthcare → 5. AED → 6. Ramps → 7. Weather → 8. Venue Language

**Two-Phase ETL Design**:

| Phase | Cell | Function | Description |
|-------|------|----------|-------------|
| **Phase 1: Dedup** | [12] | `dedup_restrooms()` | Pre-dedup by name, filter Manhattan |
| | [12] | `dedup_parks()` | Pre-dedup by name, filter Manhattan |
| | [12] | `dedup_aed()` | Pre-dedup by entity+address+floor, filter Manhattan |
| | [12] | `dedup_healthcare()` | NYS filter Manhattan + OSM GPS match (<30m) |
| | [12] | `dedup_ramps()` | Pre-dedup by ramp_id, filter Borough=1 |
| **Phase 2: Import** | [16] | `etl_restrooms()` | Import pre-deduped data to venues + restroom_profiles |
| | [20] | `etl_healthcare()` | Import pre-deduped data to venues + healthcare_profiles |
| | [18] | `etl_aed()` | Import pre-deduped data to venues + emergency_assets |
| | [22] | `etl_ramps()` | Import pre-deduped data to pedestrian_ramps |
| | [40] | `etl_weather()` | Fetch NWS API, cache to external_context_cache |
| | [42] | `etl_venue_language()` | GPS match LASS, insert to venue_language |

**Dedup Functions** (cell-12):

| Function | Dedup Key | Filter | Output |
|----------|-----------|--------|--------|
| `dedup_restrooms()` | `name.lower()` | Manhattan GPS | `restrooms_deduped` |
| `dedup_parks()` | `name.lower()` | Manhattan Borough | `parks_deduped` |
| `dedup_aed()` | `ename|address|floor` | Manhattan Borough | `aed_deduped` |
| `dedup_healthcare()` | GPS <30m (NYS priority) | Manhattan | `nys_deduped, osm_deduped` |
| `dedup_ramps()` | `ramp_id` | Borough=1 | `ramps_deduped` |

**ETL Import Functions** (cell-16, 18, 20, 22):

| Function | Source | Target Tables | Import Method |
|----------|--------|---------------|---------------|
| `etl_restrooms()` | NYC Restrooms + Parks | venues, restroom_profiles, venue_source_links | INSERT ON DUPLICATE |
| `etl_healthcare()` | OSM + NYS Health | venues, healthcare_profiles, venue_source_links | INSERT ON DUPLICATE |
| `etl_aed()` | AED Inventory | venues, emergency_assets, venue_source_links | INSERT ON DUPLICATE |
| `etl_ramps()` | Pedestrian Ramps | pedestrian_ramps | INSERT ON DUPLICATE (batch) |
| `etl_weather()` | NWS API | external_context_cache, venues | INSERT ON DUPLICATE |
| `etl_venue_language()` | LASS Ratings | venue_language | INSERT ON DUPLICATE |

**Utility Functions** (cell-12):

| Function | Purpose | Usage |
|----------|---------|-------|
| `check_row(row, required_fields)` | Check data completeness | Validate required fields |
| `validate_coords(lat, lng, bbox)` | Check coordinate validity | Verify lat/lng range |
| `dedup_check(data, key_fields)` | Generic dedup check | Remove duplicates by key |
| `fill_missing(row, defaults)` | Fill missing values | Apply default values |
| `log_import(table, imported, skipped)` | Log import details | Print statistics |

**Confidence Levels**:

| Source | Confidence | Reason |
|--------|------------|--------|
| NYS Health | 0.9 | Official government data |
| AED Inventory | 0.8 | Verified inventory |
| NYC Restrooms | 0.6 | City data, may be outdated |
| OSM Healthcare | 0.5 | Community-sourced |
| Parks Toilets | 0.3 | No coordinates available |
| LASS Ratings | 0.4 | Government evaluation data |

**Note**:
- `etl_osm` and `etl_nys` are defined for reference only - NOT executed directly. Use `etl_healthcare` for merged OSM+NYS import.
- All ETL functions use `INSERT ... ON DUPLICATE KEY UPDATE` for idempotent execution.
- Dedup preprocessing ensures clean data before database import.

In [ ]:
try:
    conn = get_conn()
    with conn.cursor() as cur: cur.execute('SELECT 1')
    conn.close()
    print('MySQL connection OK')
except Exception as e:
    print(f'MySQL connection failed: {e}')

# ============================================================
# ETL Utility Functions
# ============================================================

def check_row(row, required_fields=None):
    """check data completeness, return (is_valid, errors)"""
    errors = []
    if required_fields:
        for field in required_fields:
            val = row.get(field)
            if val is None or (isinstance(val, str) and not val.strip()):
                errors.append(f'Missing: {field}')
    return len(errors) == 0, errors

def validate_coords(lat, lng, bbox=None):
    """check if coordinates are valid and optionally within a bounding box"""
    if lat is None or lng is None:
        return False, 'Missing coordinates'
    try:
        lat, lng = float(lat), float(lng)
    except (ValueError, TypeError):
        return False, 'Invalid coordinate format'
    if not (-90 <= lat <= 90) or not (-180 <= lng <= 180):
        return False, 'Coordinates out of range'
    if bbox:
        if not (bbox['lat_min'] <= lat <= bbox['lat_max'] and bbox['lng_min'] <= lng <= bbox['lng_max']):
            return False, f'Coordinates outside bbox'
    return True, None

def dedup_check(data, key_fields, source_name='data'):
    """check for duplicate rows, return deduplicated data"""
    seen = set()
    deduped = []
    dup_count = 0
    for row in data:
        key = tuple((row.get(f, '') or '').strip().lower() for f in key_fields)
        if key in seen:
            dup_count += 1
            continue
        seen.add(key)
        deduped.append(row)
    if dup_count > 0:
        print(f'  [{source_name}] Dedup: removed {dup_count} duplicates, {len(deduped)} unique')
    return deduped

def fill_missing(row, defaults=None):
    """fill missing values"""
    if defaults is None:
        defaults = {}
    filled = 0
    for field, default_val in defaults.items():
        val = row.get(field)
        if val is None or (isinstance(val, str) and not val.strip()):
            row[field] = default_val
            filled += 1
    return row, filled

def log_import(table, imported, skipped, details=None):
    """log import details"""
    print(f'  [{table}] Imported: {imported}, Skipped: {skipped}')
    if details:
        for k, v in details.items():
            print(f'    {k}: {v}')

# ============================================================
# Dedup Preprocessing Functions (NEW)
# ============================================================

def dedup_restrooms(restrooms_data):
    """Pre-dedup NYC Restrooms by name (within source)."""
    seen = set()
    deduped = []
    dup_count = 0
    for row in restrooms_data:
        name = (row.get('Facility Name') or '').strip()
        try:
            lat, lng = float(row.get('Latitude', 0) or 0), float(row.get('Longitude', 0) or 0)
        except (ValueError, TypeError, KeyError):
            continue
        if not name or not is_manhattan(lat, lng):
            continue
        name_key = name.lower()
        if name_key in seen:
            dup_count += 1
            continue
        seen.add(name_key)
        deduped.append(row)
    print(f'  [restrooms] Dedup: {len(restrooms_data)} → {len(deduped)} unique (removed {dup_count} duplicates)')
    return deduped

def dedup_parks(parks_data):
    """Pre-dedup Parks Toilets by name (within source)."""
    seen = set()
    deduped = []
    dup_count = 0
    for row in parks_data:
        name = (row.get('Name') or '').strip()
        borough = (row.get('Borough') or '').strip()
        if not name or borough.lower() != 'manhattan':
            continue
        name_key = name.lower()
        if name_key in seen:
            dup_count += 1
            continue
        seen.add(name_key)
        deduped.append(row)
    print(f'  [parks] Dedup: {len(parks_data)} → {len(deduped)} unique (removed {dup_count} duplicates)')
    return deduped

def dedup_aed(aed_data):
    """Pre-dedup AED by entity_name + address + floor (within source)."""
    seen = set()
    deduped = []
    dup_count = 0
    for row in aed_data:
        ename = (row.get('Entity_Name') or '').strip()
        address = (row.get('Address') or '').strip()
        floor = (row.get('Floor') or '').strip()
        if not ename:
            continue
        if (row.get('Borough') or '').strip().lower() != 'manhattan':
            continue
        dedup_key = f"{ename.lower()}|{address.lower()}|{floor.lower()}"
        if dedup_key in seen:
            dup_count += 1
            continue
        seen.add(dedup_key)
        deduped.append(row)
    print(f'  [aed] Dedup: {len(aed_data)} → {len(deduped)} unique (removed {dup_count} duplicates)')
    return deduped

def dedup_healthcare(osm_features, nys_data):
    """Pre-dedup Healthcare: NYS first, OSM GPS match (<30m)."""
    GPS_THRESHOLD = 30
    
    # Step 1: Filter NYS (official, priority)
    nys_deduped = []
    for row in nys_data:
        county = (row.get('Facility County') or '').strip()
        if county != 'New York':
            continue
        name = (row.get('Facility Name') or '').strip()
        if not name:
            continue
        try:
            lat, lng = float(row.get('Facility Latitude', '')), float(row.get('Facility Longitude', ''))
        except (ValueError, TypeError, KeyError):
            continue
        if not is_manhattan(lat, lng):
            continue
        nys_deduped.append(row)
    print(f'  [healthcare] NYS filtered: {len(nys_data)} → {len(nys_deduped)} (Manhattan only)')
    
    # Step 2: Filter OSM, check GPS match with NYS
    osm_deduped = []
    osm_matched = 0
    nys_coords = []
    for row in nys_deduped:
        try:
            lat, lng = float(row.get('Facility Latitude', '')), float(row.get('Facility Longitude', ''))
            nys_coords.append((lat, lng))
        except (ValueError, TypeError, KeyError):
            pass
    
    for feat in osm_features:
        coords = feat.get('geometry', {}).get('coordinates', [])
        if len(coords) < 2:
            continue
        lng, lat = coords[0], coords[1]
        if not is_manhattan(lat, lng):
            continue
        
        # Check GPS match with NYS
        matched = False
        for nys_lat, nys_lng in nys_coords:
            dist = ((lat - nys_lat)**2 + (lng - nys_lng)**2)**0.5 * 111000
            if dist < GPS_THRESHOLD:
                osm_matched += 1
                matched = True
                break
        
        if not matched:
            osm_deduped.append(feat)
    
    print(f'  [healthcare] OSM filtered: {len(osm_features)} → {len(osm_deduped)} unique (GPS match removed {osm_matched})')
    return nys_deduped, osm_deduped

def dedup_ramps(ramps_data):
    """Pre-dedup Pedestrian Ramps by ramp_id (within source)."""
    seen = set()
    deduped = []
    dup_count = 0
    for row in ramps_data:
        if (row.get('Borough') or '').strip() != '1':
            continue
        ramp_id = (row.get('RampID') or '').strip()
        if not ramp_id:
            continue
        if ramp_id in seen:
            dup_count += 1
            continue
        seen.add(ramp_id)
        deduped.append(row)
    print(f'  [ramps] Dedup: {len(ramps_data)} → {len(deduped)} unique (removed {dup_count} duplicates)')
    return deduped

print('ETL utility functions loaded: check_row, validate_coords, dedup_check, fill_missing, log_import')
print('Dedup preprocessing functions loaded: dedup_restrooms, dedup_parks, dedup_aed, dedup_healthcare, dedup_ramps')

## Schema Rebuild: Drop and recreate all tables
## WARNING: This will DELETE all existing data

In [ ]:
try:
    conn = get_conn()
    with conn.cursor() as cur:
        # Disable foreign key checks for drop AND create
        cur.execute('SET FOREIGN_KEY_CHECKS = 0')
        
        # Drop all tables
        tables = ['venues', 'venue_source_links', 'restroom_profiles', 'healthcare_profiles', 
                  'emergency_assets', 'pedestrian_ramps', 'user_reports', 'report_confirmations',
                  'busyness_scores', 'external_context_cache', 'venue_accessibility', 'venue_language', 'venue_warnings',
                  'users', 'user_favorite_venues', 'notification_preferences', 'report_categories',
                  'busyness_forecasts', 'venue_embeddings']
        for table in tables:
            cur.execute(f'DROP TABLE IF EXISTS {table}')
        
        # Recreate schema from SQL file (FK checks still OFF)
        schema_sql = SCHEMA_PATH.read_text(encoding='utf-8')
        for stmt in schema_sql.split(';'):
            stmt = stmt.strip()
            if stmt:
                cur.execute(stmt)
        
        # Re-enable foreign key checks
        cur.execute('SET FOREIGN_KEY_CHECKS = 1')
    
    conn.commit()
    conn.close()
    print('Schema rebuilt: all tables dropped and recreated')
except Exception as e:
    print(f'Schema rebuild failed: {e}')

### Restroom — ETL

#### Deduplication conclusion

The current Restroom ETL primarily performs **within-source deduplication**, rather than full cross-source entity deduplication:

- NYC Public Restrooms: 358 Manhattan records, all with latitude and longitude. Records are deduplicated by normalized facility name.
- Parks Toilets: 129 Manhattan records, none with latitude or longitude. Records are deduplicated by normalized park/toilet name.
- Both sources are inserted into `venues`, but their IDs include the source name (`nyc_restrooms` or `parks_toilets`). Therefore, the same physical restroom can receive two different `venue_id` values.
- After normalizing case and punctuation, 37 exact-name overlaps were found between the two Manhattan datasets. These are high-confidence duplicate candidates, but they are not currently merged automatically.

The main constraint is the complete absence of GPS coordinates in Parks Toilets. Names can still be compared, but fuzzy name matching alone can create false positives because park, playground, library, and comfort-station names may partially overlap. The Parks `Location` field is free-form street text and cannot be treated as a reliable coordinate without normalization or geocoding.

A safer future cross-source strategy is:

1. Automatically merge exact normalized-name matches after checking the location text.
2. Treat fuzzy-name plus compatible street text as review candidates.
3. Geocode Parks `Location`, then merge records only when name evidence is compatible and distance is within approximately 50 metres.
4. Keep uncertain records separate and export them for manual review.


In [ ]:
def etl_restrooms(conn, restrooms_data, parks_data):
    """Import pre-deduped NYC Public Restrooms + Parks Toilets."""
    imp, skip = 0, 0
    for row in restrooms_data:
        name = (row.get('Facility Name') or '').strip()
        try:
            lat, lng = float(row['Latitude']), float(row['Longitude'])
        except (ValueError, TypeError, KeyError) as error:
            log_etl_error('nyc_restrooms', name or '<missing-name>', error)
            skip += 1
            continue
        if not name or not is_manhattan(lat, lng):
            skip += 1
            continue
        sid = source_hash(name, str(lat), str(lng))
        vid = gen_vid('nyc_restrooms', sid)
        district = gps_to_district(lat, lng)
        status_raw = (row.get('Status') or '').strip().lower()
        status = 'operational' if 'operational' in status_raw and 'not' not in status_raw else 'not_operational'
        statements = [
            ('INSERT INTO venues (venue_id, venue_type, name, latitude, longitude, borough, district, address, website, source_confidence) VALUES (%s, "restroom", %s, %s, %s, %s, %s, %s, %s, 0.600) ON DUPLICATE KEY UPDATE name = VALUES(name)',
             (vid, name, lat, lng, row.get('Location Type', ''), district, (row.get('Location') or '').strip() or None, (row.get('Website') or '').strip() or None)),
            ('INSERT INTO restroom_profiles (venue_id, restroom_type, operator, status, handicap_accessible, changing_station, additional_notes) VALUES (%s, %s, %s, %s, %s, %s, %s) ON DUPLICATE KEY UPDATE status = VALUES(status)',
             (vid, (row.get('Restroom Type') or '').strip() or None, (row.get('Operator') or '').strip() or None, status,
              True if 'accessible' in (row.get('Accessibility') or '').lower() and 'not' not in (row.get('Accessibility') or '').lower() else None,
              True if (row.get('Changing Stations') or '').strip().lower() == 'yes' else None, (row.get('Additional Notes') or '').strip() or None)),
            ('INSERT INTO venue_source_links (venue_id, source_name, source_record_id, raw_name, matched_method, match_confidence) VALUES (%s, "nyc_restrooms", %s, %s, "single_source", 0.600) ON DUPLICATE KEY UPDATE match_confidence = VALUES(match_confidence)',
             (vid, sid, name)),
        ]
        if etl_execute(conn, statements, source='nyc_restrooms', record_id=sid):
            imp += 1
        else:
            skip += 1

    for row in parks_data:
        name = (row.get('Name') or '').strip()
        borough = (row.get('Borough') or '').strip()
        if not name or borough.lower() != 'manhattan':
            skip += 1
            continue
        sid = source_hash(name, borough)
        vid = gen_vid('parks_toilets', sid)
        location_text = (row.get('Location') or '').strip() or None
        notes = f'Location: {location_text}' if location_text else None
        statements = [
            ('INSERT INTO venues (venue_id, venue_type, name, latitude, longitude, borough, district, address, source_confidence) VALUES (%s, "restroom", %s, 0, 0, %s, NULL, %s, 0.300) ON DUPLICATE KEY UPDATE name = VALUES(name)',
             (vid, name, borough, location_text)),
            ('INSERT INTO restroom_profiles (venue_id, restroom_type, operator, open_year_round, handicap_accessible, additional_notes) VALUES (%s, %s, %s, %s, %s, %s) ON DUPLICATE KEY UPDATE additional_notes = VALUES(additional_notes)',
             (vid, 'park', 'NYC Parks', True if (row.get('Open Year-Round') or '').strip().lower() == 'yes' else None,
              True if (row.get('Handicap Accessible') or '').strip().lower() == 'yes' else None, notes)),
            ('INSERT INTO venue_source_links (venue_id, source_name, source_record_id, raw_name, matched_method, match_confidence) VALUES (%s, "parks_toilets", %s, %s, "single_source", 0.300) ON DUPLICATE KEY UPDATE match_confidence = VALUES(match_confidence)',
             (vid, sid, name)),
        ]
        if etl_execute(conn, statements, source='parks_toilets', record_id=sid):
            imp += 1
        else:
            skip += 1
    return imp, skip

# ============================================================
# Execute: Dedup → Import
# ============================================================
print('=== Restrooms ETL: Dedup → Import ===')
restrooms_deduped = dedup_restrooms(restrooms)
parks_deduped = dedup_parks(parks)

conn = get_conn()
imp, skip = etl_restrooms(conn, restrooms_deduped, parks_deduped)
conn.close()
print(f'Restrooms (merged): imported={imp}, skipped={skip}')

### Healthcare — AED ETL

#### Deduplication analysis

The apparent loss of more than half of the AED dataset comes from two separate operations:

| Stage | Records | Removed |
|-------|--------:|--------:|
| Original NYC dataset | 7,373 | — |
| Manhattan borough filter | 3,393 | 3,980 non-Manhattan records |
| Current `Entity_Name + Address + Floor` deduplication | 3,279 | 114 Manhattan records |

All 3,393 Manhattan records have a facility name and parseable coordinates. Therefore, the second reduction is not caused by missing names or GPS data. It is caused by the current deduplication key being too coarse.

Many rows at the same entity and address represent distinct AED installations on different floors or at different internal locations. Examples include:

- The Museum of Modern Art: 47 rows at one address, generally representing different floor/location descriptions.
- Morgan Stanley, 1585 Broadway: 43 rows with different floors and different AED counts.
- Marymount School, 115 E 97th St: 20 rows distributed across different floors.

Only about 58 Manhattan rows are duplicate after normalizing the complete row. Alternative key results are:

| Deduplication key | Retained | Removed |
|-------------------|---------:|--------:|
| `Entity_Name + Address` | 1,775 | 1,618 |
| `Entity_Name + Address + Floor` | 3,279 | 114 |
| Normalized complete row | about 3,335 | about 58 |

**Conclusion:** the current result of 3,279 represents unique entity/address/floor combinations, preserving installation-level detail. Only 114 true duplicates (same entity, address, and floor) were removed.

Recommended model:

1. Keep one `venues` record per normalized entity and address.
2. Store each floor/location record in a separate `aed_installations` child table linked to the venue.
3. Deduplicate installation rows using normalized entity, address, floor, location type and relevant count fields, or remove only normalized full-row duplicates.
4. Derive site-level totals carefully; do not blindly sum repeated reports because some rows may be updated submissions rather than separate devices.

Until the child-table model is implemented, `Entity_Name + Address + Floor` is safer than the current key, but it can still merge records where the floor is blank or represented inconsistently.

In [ ]:
def etl_aed(conn, data):
    """Import pre-deduped AED Inventory."""
    imp, skip = 0, 0
    for row in data:
        ename = (row.get('Entity_Name') or '').strip()
        if not ename:
            skip += 1
            continue
        try:
            lat, lng = float(row['Latitude']), float(row['Longitude'])
        except (ValueError, TypeError, KeyError) as error:
            log_etl_error('aed_inventory', ename, error)
            skip += 1
            continue
        if (row.get('Borough') or '').strip().lower() != 'manhattan':
            skip += 1
            continue
        address = (row.get('Address') or '').strip()
        floor = (row.get('Floor') or '').strip()
        sid = source_hash(ename, address, floor)
        vid = gen_vid('aed_inventory', sid)
        district = gps_to_district(lat, lng)
        last_updated = None
        last_updated_str = (row.get('Last Updated') or '').strip()
        if last_updated_str:
            try:
                last_updated = datetime.datetime.strptime(last_updated_str, '%m/%d/%Y').strftime('%Y-%m-%d')
            except ValueError as error:
                log_etl_error('aed_inventory', sid, error)
        statements = [
            ('INSERT INTO venues (venue_id, venue_type, name, latitude, longitude, borough, district, address, source_confidence) VALUES (%s, "emergencyasset", %s, %s, %s, %s, %s, %s, 0.800) ON DUPLICATE KEY UPDATE name = VALUES(name)',
             (vid, f'{ename} AED', lat, lng, (row.get('Borough') or '').strip() or None, district, address or None)),
            ('INSERT INTO emergency_assets (venue_id, asset_type, floor, location_type, aed_count, trained_people_count, community_district, council_district, last_updated) VALUES (%s, "aed", %s, %s, %s, %s, %s, %s, %s) ON DUPLICATE KEY UPDATE aed_count = VALUES(aed_count)',
             (vid, (row.get('Floor') or '').strip() or None, (row.get('Location Type') or '').strip() or None, safe_int(row.get('AED_NumAeds')), safe_int(row.get('AED_NumPersonTrained')), (row.get('Community_District') or '').strip() or None, (row.get('Council_District') or '').strip() or None, last_updated)),
            ('INSERT INTO venue_source_links (venue_id, source_name, source_record_id, raw_name, raw_location_text, matched_method, match_confidence) VALUES (%s, "aed_inventory", %s, %s, %s, "single_source", 0.800) ON DUPLICATE KEY UPDATE match_confidence = VALUES(match_confidence)',
             (vid, sid, ename, address)),
        ]
        if etl_execute(conn, statements, source='aed_inventory', record_id=sid):
            imp += 1
        else:
            skip += 1
    return imp, skip

# ============================================================
# Execute: Dedup → Import
# ============================================================
print('=== AED ETL: Dedup → Import ===')
aed_deduped = dedup_aed(aed)

conn = get_conn()
imp, skip = etl_aed(conn, aed_deduped)
conn.close()
print(f'AED Inventory: imported={imp}, skipped={skip}')

### Healthcare — hospital/clinic/pharmacy ETL
 
| Metric | Count |
|--------|------:|
| Manhattan by County (`New York`) | 454 |
| Missing `Facility Name` | 0 |
| Missing / unparseable coordinates | 13 |
| Valid (name + coords) | **441** |

13 records have empty `Facility Latitude` / `Facility Longitude` fields (e.g. Universal Care, Inc., The Apsley). These are skipped by the `float()` parse in the ETL.

**Note**: `etl_healthcare()` has inline NYS import logic (does NOT call `etl_nys()`).

#### Deduplication strategy

Healthcare uses both **within-source filtering** and **cross-source deduplication** before records are inserted into the database.

1. **NYS Health is treated as the authoritative source.** NYS records are restricted to `Facility County == "New York"`, require a facility name and parseable coordinates, and must fall inside the Manhattan bounding box.
2. **NYS records are retained first.** Their valid coordinates are collected as the reference set for cross-source matching.
3. **OSM Healthcare records are checked against NYS coordinates.** If an OSM feature is within 30 metres of any retained NYS facility, it is treated as the same physical healthcare venue and the OSM record is removed.
4. **Unmatched OSM records are retained.** OSM contributes facilities that are not represented in NYS, including categories such as clinics, pharmacies, dentists and laboratories.
5. **Database writes remain idempotent.** Source-based deterministic `venue_id` values and `INSERT ... ON DUPLICATE KEY UPDATE` prevent repeated notebook runs from creating additional rows for the same retained source record.

This is therefore a **NYS-priority cross-source GPS deduplication strategy**, not only table-internal deduplication.

Current limitations:

- The 30-metre rule uses coordinates only; it does not verify name, address or healthcare category similarity.
- Two unrelated facilities in the same building may be incorrectly merged.
- Duplicate NYS records at nearby coordinates are not comprehensively entity-matched by name and address.
- NYS records with missing coordinates cannot participate in cross-source matching and are skipped.
- OSM records with inaccurate coordinates may either remain duplicated or be matched to the wrong NYS facility.

A stronger future matcher should combine distance, normalized name, address and facility category, while preserving NYS as the preferred authoritative record.

In [ ]:
def etl_healthcare(conn, nys_data, osm_data):
    """Import pre-deduped NYS and OSM healthcare data."""
    imp, skip = 0, 0
    for row in nys_data:
        county = (row.get('Facility County') or '').strip()
        name = (row.get('Facility Name') or '').strip()
        if county != 'New York' or not name:
            continue
        try:
            lat, lng = float(row.get('Facility Latitude', '')), float(row.get('Facility Longitude', ''))
        except (ValueError, TypeError) as error:
            log_etl_error('nys_health', name, error)
            skip += 1
            continue
        if not is_manhattan(lat, lng):
            continue
        fid = (row.get('Facility ID') or '').strip()
        sid = fid or source_hash(name, str(lat), str(lng))
        vid = gen_vid('nys_health', sid)
        district = gps_to_district(lat, lng)
        address = ', '.join(filter(None, ((row.get(field) or '').strip() for field in ['Facility Address 1', 'Facility Address 2']))) or None
        desc = (row.get('Description') or '').strip().lower()
        htype = 'hospital' if 'hospital' in desc else 'clinic'
        statements = [
            ('INSERT INTO venues (venue_id, venue_type, name, latitude, longitude, borough, district, address, phone, website, source_confidence) VALUES (%s, "healthcare", %s, %s, %s, %s, %s, %s, %s, %s, 0.900) ON DUPLICATE KEY UPDATE name = VALUES(name)',
             (vid, name, lat, lng, 'Manhattan', district, address, (row.get('Facility Phone Number') or '').strip() or None, (row.get('Facility Website') or '').strip() or None)),
            ('INSERT INTO healthcare_profiles (venue_id, facility_external_id, facility_type, healthcare_category, operator_name, ownership_type, official_source_priority) VALUES (%s, %s, %s, %s, %s, %s, 1) ON DUPLICATE KEY UPDATE operator_name = VALUES(operator_name)',
             (vid, fid, (row.get('Short Description') or '').strip() or None, htype, (row.get('Operator Name') or '').strip() or None, (row.get('Ownership Type') or '').strip() or None)),
            ('INSERT INTO venue_source_links (venue_id, source_name, source_record_id, raw_name, raw_location_text, matched_method, match_confidence) VALUES (%s, "nys_health", %s, %s, %s, "single_source", 0.900) ON DUPLICATE KEY UPDATE match_confidence = VALUES(match_confidence)',
             (vid, sid, name, address)),
        ]
        if etl_execute(conn, statements, source='nys_health', record_id=sid):
            imp += 1
        else:
            skip += 1

    for feat in osm_data:
        coords = feat.get('geometry', {}).get('coordinates', [])
        if len(coords) < 2:
            skip += 1
            continue
        lng, lat = coords[0], coords[1]
        if not is_manhattan(lat, lng):
            skip += 1
            continue
        props = feat.get('properties', {})
        name = (props.get('name') or '').strip()
        osm_id = (props.get('@id') or '').strip()
        sid = osm_id or source_hash(name or 'unknown', str(lat), str(lng))
        vid = gen_vid('osm', sid)
        district = gps_to_district(lat, lng)
        htype = OSM_CATEGORY_MAP.get((props.get('healthcare') or '').strip().lower()) or OSM_CATEGORY_MAP.get((props.get('amenity') or '').strip().lower()) or 'healthcare'
        # Skip unwanted types: laboratory and unknown/fallback
        ALLOWED_OSM_TYPES = {'hospital', 'clinic', 'pharmacy', 'dentist'}
        if htype not in ALLOWED_OSM_TYPES:
            skip += 1
            continue
        hn, st = props.get('addr:housenumber', ''), props.get('addr:street', '')
        address = f'{hn} {st}' if hn and st else (st or None)
        statements = [
            ('INSERT INTO venues (venue_id, venue_type, name, latitude, longitude, borough, district, address, phone, website, opening_hours, source_confidence) VALUES (%s, "healthcare", %s, %s, %s, %s, %s, %s, %s, %s, %s, 0.500) ON DUPLICATE KEY UPDATE name = VALUES(name)',
             (vid, name or None, lat, lng, (props.get('addr:city') or '').strip() or None, district, address, (props.get('phone') or props.get('contact:phone') or '').strip() or None, (props.get('website') or '').strip() or None, (props.get('opening_hours') or '').strip() or None)),
            ('INSERT INTO healthcare_profiles (venue_id, healthcare_category, facility_type, healthcare_speciality) VALUES (%s, %s, %s, %s) ON DUPLICATE KEY UPDATE healthcare_category = VALUES(healthcare_category)',
             (vid, htype, htype, (props.get('healthcare:speciality') or '').strip() or None)),
            ('INSERT INTO venue_source_links (venue_id, source_name, source_record_id, raw_name, matched_method, match_confidence) VALUES (%s, "osm", %s, %s, "single_source", 0.500) ON DUPLICATE KEY UPDATE match_confidence = VALUES(match_confidence)',
             (vid, sid, name or 'unknown')),
        ]
        if etl_execute(conn, statements, source='osm_healthcare', record_id=sid):
            imp += 1
        else:
            skip += 1
    return imp, skip

# ============================================================
# Execute: Dedup → Import
# ============================================================
print('=== Healthcare ETL: Dedup → Import ===')
nys_deduped, osm_deduped = dedup_healthcare(osm_features, nys)

conn = get_conn()
imp, skip = etl_healthcare(conn, nys_deduped, osm_deduped)
conn.close()
print(f'Healthcare (merged): imported={imp}, skipped={skip}')

### Accessibility — ETL


In [ ]:
def etl_ramps(conn, data):
    """Import pre-deduped Pedestrian Ramps in atomic batches."""
    sql = 'INSERT INTO pedestrian_ramps (ramp_id, corner_id, latitude, longitude, borough, district, on_street, cross_street_1, cross_street_2, ramp_width, ramp_slope, dws_condition, ponding, obstacles_ramp, obstacles_landing) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s) ON DUPLICATE KEY UPDATE dws_condition = VALUES(dws_condition)'
    imp, skip, batch = 0, 0, []
    for row in data:
        if (row.get('Borough') or '').strip() != '1':
            skip += 1
            continue
        geom = (row.get('the_geom') or '').strip()
        match = re.match(r'POINT\s*\(\s*(-?[\d.]+)\s+(-?[\d.]+)\s*\)', geom)
        ramp_id = (row.get('RampID') or '').strip()
        if not match or not ramp_id:
            skip += 1
            continue
        try:
            lng, lat = float(match.group(1)), float(match.group(2))
        except (ValueError, TypeError) as error:
            log_etl_error('pedestrian_ramps', ramp_id, error)
            skip += 1
            continue
        district = gps_to_district(lat, lng)
        batch.append((ramp_id, (row.get('CornerID') or '').strip() or None, lat, lng, 'Manhattan', district,
            (row.get('Ramp_OnStreet') or '').strip() or None, (row.get('StName1') or '').strip() or None, (row.get('StName2') or '').strip() or None,
            safe_dec(row.get('RAMP_WIDTH')), safe_dec(row.get('RAMP_RUNNING_SLOPE_TOTAL')),
            (row.get('DWS_CONDITIONS') or '').strip() or None, (row.get('PONDING') or '').strip() or None,
            (row.get('OBSTACLES_RAMP') or '').strip() or None, (row.get('OBSTACLES_LANDING') or '').strip() or None))
        if len(batch) >= 1000:
            success, failed = etl_executemany(conn, sql, batch, source='pedestrian_ramps', record_id=f'batch-ending-{ramp_id}')
            imp += success
            skip += failed
            batch = []
    if batch:
        success, failed = etl_executemany(conn, sql, batch, source='pedestrian_ramps', record_id='final-batch')
        imp += success
        skip += failed
    return imp, skip

# ============================================================
# Execute: Dedup → Import
# ============================================================
print('=== Ramps ETL: Dedup → Import ===')
ramps_deduped = dedup_ramps(ramps)

conn = get_conn()
imp, skip = etl_ramps(conn, ramps_deduped)
conn.close()
print(f'Pedestrian Ramps: imported={imp}, skipped={skip}')

---
## Part 6: Import Verification

## Data overview after import:
 

In [ ]:
conn = get_conn()
tables = ['venues', 'venue_source_links', 'restroom_profiles', 'healthcare_profiles', 'emergency_assets', 'pedestrian_ramps', 'user_reports', 'busyness_scores']
print(f'{"Table":<30} {"Rows":>10}')
print('-' * 42)
with conn.cursor() as cur:
    for table in tables:
        cur.execute(f'SELECT COUNT(*) FROM {table}')
        print(f'{table:<30} {cur.fetchone()[0]:>10}')

# Data source breakdown
print('\n--- Data Source Breakdown ---')
with conn.cursor() as cur:
    cur.execute("SELECT venue_type, COUNT(*) FROM venues GROUP BY venue_type")
    for row in cur.fetchall():
        print(f'{row[0]:<30} {row[1]:>10}')

conn.close()

---
## Part 7: Schema Update (Align to API)

Apply schema changes to align the database structure with the backend API (based on `origin/eq_sprint1` `mock_data.py`).

Changes:
1. 3 new tables**: `venue_accessibility`, `venue_language`, `venue_warnings`
2. venues new columns**: `photos` (JSON), `rating` (DECIMAL(3,2))
3. user_reports new columns**: `anonymous`, `description`, `photos`
4. report_confirmations new column**: `language`
5. busyness_scores new columns**: `forecast_1h`, `forecast_4h`, `forecast_8h`

Execution: SQL defined in `legacy schema updates` variable, executed one statement at a time via `split(';')`, skipping errors automatically.

In [ ]:
# Schema updates are consolidated in the Schema Migration section below.
# This cell is kept for backward compatibility
# Run the MIGRATIONS definition and apply_migrations execution cells below.
print('Schema updates moved to the consolidated MIGRATIONS section')
print('Run the migration definition and execution cells below')


---
## Part 8: Updated Table Structure

In [ ]:
conn = get_conn()
with conn.cursor() as cur:
    cur.execute('SHOW TABLES')
    tables = [r[0] for r in cur.fetchall()]
    print(f'Total tables: {len(tables)}')
    print('Tables:', sorted(tables))
    for table in ['venue_accessibility', 'venue_language', 'venue_warnings']:
        if table in tables:
            cur.execute(f'DESCRIBE {table}')
            cols = [r[0] for r in cur.fetchall()]
            print(f'  {table}: {cols}')
conn.close()

---
## Part 10: Backend API Schema Alignment

based on origin/eq_sprint1 / mock_data.py, optimized for the database structure and data quality insights.

In [ ]:
# Backend API schema updates are consolidated in the Schema Migration section below.
# This cell is kept for backward compatibility
# Run the MIGRATIONS definition and apply_migrations execution cells below.
print('Backend API schema updates moved to the consolidated MIGRATIONS section')
print('Run the migration definition and execution cells below')


---
## Part 11: table completation validation

In [ ]:
conn = get_conn()
with conn.cursor() as cur:
    cur.execute('SHOW TABLES')
    tables = [r[0] for r in cur.fetchall()]
    print(f'Total tables: {len(tables)}')
    
    # Verify venues has new columns
    cur.execute('DESCRIBE venues')
    venue_cols = [r[0] for r in cur.fetchall()]
    print(f'\nvenues columns ({len(venue_cols)}):', venue_cols)
    
    # Verify user_reports has new columns
    cur.execute('DESCRIBE user_reports')
    report_cols = [r[0] for r in cur.fetchall()]
    print(f'\nuser_reports columns ({len(report_cols)}):', report_cols)
    
    # Verify new tables exist
    for table in ['venue_accessibility', 'venue_language', 'venue_warnings']:
        if table in tables:
            cur.execute(f'DESCRIBE {table}')
            cols = [r[0] for r in cur.fetchall()]
            print(f'\n{table} ({len(cols)}):', cols)
conn.close()

---
## Part 12: Fix Plan Execution

based on fix_plan.md，carry out Schema optimalization.

In [ ]:
# Schema Migration - single source of truth
# Each entry has a stable name and enough metadata for preflight checks.
MIGRATIONS = [{'name': 'modify venues.venue_type',
  'kind': 'always',
  'sql': 'ALTER TABLE venues MODIFY COLUMN venue_type ENUM(\n'
         "    'restroom', 'healthcare', 'emergencyasset',\n"
         "    'clinic', 'pharmacy', 'hospital', 'dentist', 'laboratory'\n"
         ') NOT NULL'},
 {'name': 'add venues.language_tags',
  'kind': 'column',
  'table': 'venues',
  'column': 'language_tags',
  'sql': 'ALTER TABLE venues ADD COLUMN language_tags JSON AFTER borough'},
 {'name': 'add venues.primary_language',
  'kind': 'column',
  'table': 'venues',
  'column': 'primary_language',
  'sql': 'ALTER TABLE venues ADD COLUMN primary_language VARCHAR(10) AFTER language_tags'},
 {'name': 'add venues.secondary_language',
  'kind': 'column',
  'table': 'venues',
  'column': 'secondary_language',
  'sql': 'ALTER TABLE venues ADD COLUMN secondary_language VARCHAR(10) AFTER primary_language'},
 {'name': 'add venues.accessible_status',
  'kind': 'column',
  'table': 'venues',
  'column': 'accessible_status',
  'sql': "ALTER TABLE venues ADD COLUMN accessible_status ENUM('full_access', 'partial', "
         "'step_free_route_only', 'none') DEFAULT 'none' AFTER secondary_language"},
 {'name': 'add venues.accessibility_features',
  'kind': 'column',
  'table': 'venues',
  'column': 'accessibility_features',
  'sql': 'ALTER TABLE venues ADD COLUMN accessibility_features JSON AFTER accessible_status'},
 {'name': 'add venues.active_warning',
  'kind': 'column',
  'table': 'venues',
  'column': 'active_warning',
  'sql': 'ALTER TABLE venues ADD COLUMN active_warning BOOLEAN DEFAULT FALSE AFTER accessibility_features'},
 {'name': 'add venues.photos',
  'kind': 'column',
  'table': 'venues',
  'column': 'photos',
  'sql': 'ALTER TABLE venues ADD COLUMN photos JSON AFTER opening_hours'},
 {'name': 'add venues.rating',
  'kind': 'column',
  'table': 'venues',
  'column': 'rating',
  'sql': 'ALTER TABLE venues ADD COLUMN rating DECIMAL(3,2) AFTER photos'},
 {'name': 'add venues.weather_risk',
  'kind': 'column',
  'table': 'venues',
  'column': 'weather_risk',
  'sql': "ALTER TABLE venues ADD COLUMN weather_risk ENUM('low', 'medium', 'high') DEFAULT 'low' AFTER "
         'rating'},
 {'name': 'add venues.district',
  'kind': 'column',
  'table': 'venues',
  'column': 'district',
  'sql': "ALTER TABLE venues ADD COLUMN district VARCHAR(32) DEFAULT NULL AFTER borough"},
 {'name': 'add user_reports.anonymous',
  'kind': 'column',
  'table': 'user_reports',
  'column': 'anonymous',
  'sql': 'ALTER TABLE user_reports ADD COLUMN anonymous BOOLEAN DEFAULT FALSE AFTER accuracy_meters'},
 {'name': 'add user_reports.description',
  'kind': 'column',
  'table': 'user_reports',
  'column': 'description',
  'sql': 'ALTER TABLE user_reports ADD COLUMN description TEXT AFTER anonymous'},
 {'name': 'add user_reports.photos',
  'kind': 'column',
  'table': 'user_reports',
  'column': 'photos',
  'sql': 'ALTER TABLE user_reports ADD COLUMN photos JSON AFTER description'},
 {'name': 'add user_reports.reported_by',
  'kind': 'column',
  'table': 'user_reports',
  'column': 'reported_by',
  'sql': "ALTER TABLE user_reports ADD COLUMN reported_by VARCHAR(50) DEFAULT 'anonymous' AFTER photos"},
 {'name': 'add user_reports.expires_in_minutes',
  'kind': 'column',
  'table': 'user_reports',
  'column': 'expires_in_minutes',
  'sql': 'ALTER TABLE user_reports ADD COLUMN expires_in_minutes INT DEFAULT 120 AFTER status'},
 {'name': 'add user_reports.default_language',
  'kind': 'column',
  'table': 'user_reports',
  'column': 'default_language',
  'sql': 'ALTER TABLE user_reports ADD COLUMN default_language VARCHAR(10) AFTER expires_in_minutes'},
 {'name': 'add user_reports.fallback_language',
  'kind': 'column',
  'table': 'user_reports',
  'column': 'fallback_language',
  'sql': 'ALTER TABLE user_reports ADD COLUMN fallback_language VARCHAR(10) AFTER default_language'},
 {'name': 'add report_confirmations.language',
  'kind': 'column',
  'table': 'report_confirmations',
  'column': 'language',
  'sql': 'ALTER TABLE report_confirmations ADD COLUMN language VARCHAR(10) AFTER action'},
 {'name': 'add busyness_scores.forecast_1h',
  'kind': 'column',
  'table': 'busyness_scores',
  'column': 'forecast_1h',
  'sql': 'ALTER TABLE busyness_scores ADD COLUMN forecast_1h JSON AFTER estimated_wait_minutes'},
 {'name': 'add emergency_assets unique constraint',
  'kind': 'index',
  'table': 'emergency_assets',
  'column': 'uq_emergency_asset_natural',
  'sql': 'ALTER TABLE emergency_assets ADD UNIQUE KEY uq_emergency_asset_natural '
         '(venue_id, floor, location_type)'},
 {'name': 'add pedestrian_ramps.district',
  'kind': 'column',
  'table': 'pedestrian_ramps',
  'column': 'district',
  'sql': "ALTER TABLE pedestrian_ramps ADD COLUMN district VARCHAR(32) DEFAULT NULL AFTER borough"},
 {'name': 'create venue_accessibility',
  'kind': 'table',
  'table': 'venue_accessibility',
  'sql': 'CREATE TABLE IF NOT EXISTS venue_accessibility (\n'
         '    venue_id VARCHAR(36) PRIMARY KEY,\n'
         '    wheelchair_friendly BOOLEAN DEFAULT FALSE,\n'
         '    step_free_route BOOLEAN DEFAULT FALSE,\n'
         '    accessible_toilet BOOLEAN DEFAULT FALSE,\n'
         '    entrance_width_cm INT,\n'
         '    FOREIGN KEY (venue_id) REFERENCES venues(venue_id) ON DELETE CASCADE\n'
         ')'},
 {'name': 'create venue_language',
  'kind': 'table',
  'table': 'venue_language',
  'sql': 'CREATE TABLE IF NOT EXISTS venue_language (\n'
         '    venue_id VARCHAR(36) PRIMARY KEY,\n'
         '    language_tag JSON,\n'
         "    language_support_level ENUM('full', 'partial', 'none') DEFAULT 'none',\n"
         '    chatbot_enabled BOOLEAN DEFAULT FALSE,\n'
         '    chatbot_welcoming_message TEXT,\n'
         '    FOREIGN KEY (venue_id) REFERENCES venues(venue_id) ON DELETE CASCADE\n'
         ')'},
 {'name': 'create venue_warnings',
  'kind': 'table',
  'table': 'venue_warnings',
  'sql': 'CREATE TABLE IF NOT EXISTS venue_warnings (\n'
         '    venue_id VARCHAR(36) PRIMARY KEY,\n'
         '    active_warning BOOLEAN DEFAULT FALSE,\n'
         '    warning_detail TEXT,\n'
         '    wait_alert BOOLEAN DEFAULT FALSE,\n'
         '    replacement_suggestion JSON,\n'
         '    FOREIGN KEY (venue_id) REFERENCES venues(venue_id) ON DELETE CASCADE\n'
         ')'}]

print(f'Migrations loaded: {len(MIGRATIONS)}')

### Execution Logic

```mermaid
graph TD
    A[Read MIGRATIONS] --> B[Split by ;]
    B --> C{Execute SQL}
    C -->|Success| D[Next statement]
    C -->|Fail: column/table exists| E[Print Skip → Continue]
    C -->|Fail: other error| F[Print Skip → Continue]
    D --> G{More statements?}
    E --> G
    G -->|Yes| C
    G -->|No| H[conn.commit]
```

| Helper Function | Purpose |
|-----------------|--------|
| `column_exists(conn, table, col)` | Check if column exists to avoid duplicate ADD |
| `table_exists(conn, table)` | Check if table exists to avoid duplicate CREATE |

**Skip reason**: `ALTER TABLE ADD COLUMN` errors on existing columns — this is expected, skip and continue.

In [ ]:
DUPLICATE_OBJECT_CODES = {1050, 1060}

def migration_is_applied(conn, migration):
    if migration['kind'] == 'column':
        return column_exists(conn, migration['table'], migration['column'])
    if migration['kind'] == 'table':
        return table_exists(conn, migration['table'])
    if migration['kind'] == 'index':
        with conn.cursor() as cur:
            cur.execute(
                "SELECT COUNT(*) FROM information_schema.STATISTICS "
                "WHERE TABLE_SCHEMA = %s AND TABLE_NAME = %s AND INDEX_NAME = %s",
                (MYSQL_CONFIG['database'], migration['table'], migration['column']),
            )
            return cur.fetchone()[0] > 0
    return False

def apply_migrations(conn, migrations):
    applied, skipped = 0, 0
    for migration in migrations:
        name = migration['name']
        if migration_is_applied(conn, migration):
            print(f'  SKIP already applied: {name}')
            skipped += 1
            continue
        try:
            with conn.cursor() as cur:
                cur.execute(migration['sql'])
            conn.commit()
            print(f'  APPLIED: {name}')
            applied += 1
        except pymysql.MySQLError as error:
            conn.rollback()
            error_code = error.args[0] if error.args else None
            if error_code in DUPLICATE_OBJECT_CODES:
                print(f'  SKIP duplicate object: {name} ({error_code})')
                skipped += 1
                continue
            raise RuntimeError(f'Migration failed: {name}: {error}') from error
    return applied, skipped

conn = get_conn()
try:
    applied, skipped = apply_migrations(conn, MIGRATIONS)
    print(f'Migration result: applied={applied}, skipped={skipped}')
finally:
    conn.close()

---
## Part 13: Weather ETL — NWS API Integration

**Data Source**: [weather.gov](https://api.weather.gov) (National Weather Service)  
**API Key**: Not required (public API, requires `User-Agent` header)  
**Target Table**: `external_context_cache`  
**Cache Types**: `weather_current` (current conditions), `weather_forecast` (hourly forecast)  
**Refresh**: Current weather → 1 hour TTL; Forecast → 3 hour TTL  
**Coverage**: Manhattan — resolved via NWS grid system (`/points/{lat},{lng}`)

**API Endpoints Used**:

| Endpoint | Purpose | Response |
|----------|---------|----------|
| `GET /points/{lat},{lng}` | Resolve lat/lng → NWS grid cell | gridId, gridX, gridY, forecast URLs |
| `GET /gridpoints/{id}/{x},{y}/forecast` | 7-day forecast (14 periods) | temperature, wind, shortForecast |
| `GET /stations/{id}/observations/latest` | Current conditions | temperature, humidity, heatIndex, windSpeed |

**Manhattan Grid Reference**: OKX (Upton, NY) office, grid points vary by venue location.

**Schema**: Data stored as JSON in `external_context_cache.payload_json`, keyed by `context_type + request_key`.

In [ ]:
import requests
import time

WEATHER_API_BASE = 'https://api.weather.gov'
WEATHER_HEADERS = {
    'User-Agent': 'ClearPath/1.0 (clearpath-team6@example.com)',
    'Accept': 'application/geo+json'
}
# Manhattan reference point: Times Square
MANHATTAN_LAT, MANHATTAN_LNG = 40.758, -73.985

def test_weather_api():
    """Test weather.gov API connectivity and endpoint availability."""
    results = {}
    
    # Test 1: Base endpoint
    try:
        r = requests.get(f'{WEATHER_API_BASE}/', headers=WEATHER_HEADERS, timeout=10)
        results['base'] = {'status': r.status_code, 'ok': r.status_code == 200}
        print(f'  [1] Base endpoint:       {r.status_code} {"OK" if r.ok else "FAIL"}')
    except Exception as e:
        results['base'] = {'status': 0, 'ok': False, 'error': str(e)}
        print(f'  [1] Base endpoint:       FAIL ({e})')
    
    # Test 2: Points API (lat/lng \u2192 grid)
    try:
        r = requests.get(f'{WEATHER_API_BASE}/points/{MANHATTAN_LAT},{MANHATTAN_LNG}',
                         headers=WEATHER_HEADERS, timeout=10)
        data = r.json() if r.ok else {}
        grid_id = data.get('properties', {}).get('gridId', 'N/A')
        results['points'] = {'status': r.status_code, 'ok': r.ok, 'gridId': grid_id}
        print(f'  [2] Points (grid resolve): {r.status_code} {"OK" if r.ok else "FAIL"} gridId={grid_id}')
    except Exception as e:
        results['points'] = {'status': 0, 'ok': False, 'error': str(e)}
        print(f'  [2] Points (grid resolve): FAIL ({e})')
    
    # Test 3: Forecast endpoint
    try:
        forecast_url = data.get('properties', {}).get('forecast', '')
        if forecast_url:
            r = requests.get(forecast_url, headers=WEATHER_HEADERS, timeout=10)
            periods = r.json().get('properties', {}).get('periods', []) if r.ok else []
            results['forecast'] = {'status': r.status_code, 'ok': r.ok, 'periods': len(periods)}
            print(f'  [3] Forecast:            {r.status_code} {"OK" if r.ok else "FAIL"} periods={len(periods)}')
            if periods:
                p = periods[0]
                print(f'       Latest: {p.get("name")} \u2192 {p.get("temperature")}\u00b0{p.get("temperatureUnit")}, {p.get("shortForecast")}')
        else:
            results['forecast'] = {'status': 0, 'ok': False, 'error': 'No forecast URL'}
            print(f'  [3] Forecast:            SKIP (no URL)')
    except Exception as e:
        results['forecast'] = {'status': 0, 'ok': False, 'error': str(e)}
        print(f'  [3] Forecast:            FAIL ({e})')
    
    # Test 4: Observation stations
    try:
        stations_url = data.get('properties', {}).get('observationStations', '')
        if stations_url:
            r = requests.get(stations_url, headers=WEATHER_HEADERS, timeout=10)
            stations = r.json().get('features', []) if r.ok else []
            nearest = stations[0]['properties']['stationIdentifier'] if stations else 'N/A'
            results['stations'] = {'status': r.status_code, 'ok': r.ok, 'count': len(stations), 'nearest': nearest}
            print(f'  [4] Stations:            {r.status_code} {"OK" if r.ok else "FAIL"} count={len(stations)}, nearest={nearest}')
        else:
            results['stations'] = {'status': 0, 'ok': False, 'error': 'No stations URL'}
            print(f'  [4] Stations:            SKIP (no URL)')
    except Exception as e:
        results['stations'] = {'status': 0, 'ok': False, 'error': str(e)}
        print(f'  [4] Stations:            FAIL ({e})')
    
    # Test 5: Current observation (latest)
    try:
        if nearest and nearest != 'N/A':
            obs_url = f'{WEATHER_API_BASE}/stations/{nearest}/observations/latest'
            r = requests.get(obs_url, headers=WEATHER_HEADERS, timeout=10)
            props = r.json().get('properties', {}) if r.ok else {}
            temp_c = props.get('temperature', {}).get('value')
            humidity = props.get('relativeHumidity', {}).get('value')
            desc = props.get('textDescription', 'N/A')
            results['observation'] = {'status': r.status_code, 'ok': r.ok, 'temp_c': temp_c, 'humidity': humidity}
            print(f'  [5] Current obs ({nearest}): {r.status_code} {"OK" if r.ok else "FAIL"}')
            print(f'       Temp: {temp_c}\u00b0C, Humidity: {humidity}%, Conditions: {desc}')
        else:
            results['observation'] = {'status': 0, 'ok': False, 'error': 'No station'}
            print(f'  [5] Current obs:         SKIP (no station)')
    except Exception as e:
        results['observation'] = {'status': 0, 'ok': False, 'error': str(e)}
        print(f'  [5] Current obs:         FAIL ({e})')
    
    # Summary
    ok_count = sum(1 for v in results.values() if v.get('ok'))
    total = len(results)
    print(f'\n  Summary: {ok_count}/{total} endpoints accessible')
    return results

print('=== Weather.gov API Connectivity Test ===')
print(f'Base URL: {WEATHER_API_BASE}')
print(f'Reference: Manhattan ({MANHATTAN_LAT}, {MANHATTAN_LNG})\n')
api_results = test_weather_api()
all_ok = all(v.get('ok') for v in api_results.values())
print(f'\nResult: {"ALL PASS" if all_ok else "PARTIAL FAIL"} \u2014 API is {"ready for ETL" if all_ok else "check failed endpoints"}')

In [ ]:
def fetch_current_weather(station_url=None):
    """Fetch current weather from NWS station. Returns dict with temp, humidity, wind, description, risk_level."""
    if station_url is None:
        station_url = f'{WEATHER_API_BASE}/stations/KNYC'  # Central Park
    r = requests.get(f'{station_url}/observations/latest', headers=WEATHER_HEADERS, timeout=10)
    r.raise_for_status()
    props = r.json().get('properties', {})
    return {
        'temperature_c': props.get('temperature', {}).get('value'),
        'humidity_pct': props.get('relativeHumidity', {}).get('value'),
        'wind_speed_kmh': props.get('windSpeed', {}).get('value'),
        'description': props.get('textDescription', ''),
    }

def classify_weather_risk(current):
    """Classify weather risk for busyness prediction: low | medium | high."""
    temp = current.get('temperature_c')
    wind = current.get('wind_speed_kmh')
    desc = (current.get('description') or '').lower()

    # High risk: extreme heat, high wind, severe weather
    if temp and temp > 38: return 'high'
    if wind and wind > 50: return 'high'
    if any(kw in desc for kw in ['heavy', 'thunderstorm', 'blizzard', 'ice', 'snow', 'freezing']):
        return 'high'

    # Medium risk: moderate heat, rain, moderate wind
    if temp and temp > 33: return 'medium'
    if wind and wind > 30: return 'medium'
    if any(kw in desc for kw in ['rain', 'showers', 'drizzle', 'fog', 'mist', 'windy']):
        return 'medium'

    return 'low'

def etl_weather(conn):
    """Minimal Weather ETL: fetch current conditions, classify risk, cache 1 record."""
    print('  [1] Fetching current weather...')
    try:
        current = fetch_current_weather()
        risk = classify_weather_risk(current)
        print(f'       Temp: {current["temperature_c"]}C, Wind: {current["wind_speed_kmh"]}km/h')
        print(f'       Conditions: {current["description"]}')
        print(f'       Risk level: {risk}')
    except Exception as e:
        print(f'       FAIL: {e}')
        return 0, 1

    # Cache single record
    print('  [2] Caching weather data...')
    payload = {
        'temperature_c': current['temperature_c'],
        'humidity_pct': current['humidity_pct'],
        'wind_speed_kmh': current['wind_speed_kmh'],
        'description': current['description'],
        'risk_level': risk,
    }
    statement = (
        "INSERT INTO external_context_cache"
        " (context_type, request_key, payload_json, valid_from, expires_at)"
        " VALUES (%s, %s, %s, NOW(), DATE_ADD(NOW(), INTERVAL 1 HOUR))"
        " ON DUPLICATE KEY UPDATE payload_json = VALUES(payload_json),"
        " valid_from = NOW(), expires_at = DATE_ADD(NOW(), INTERVAL 1 HOUR)",
        ('weather_current', 'weather:manhattan', json.dumps(payload, default=str)),
    )
    if etl_execute(conn, statement, source='weather', record_id='weather:manhattan'):
        print('       Cached: weather:manhattan')
        return 1, 0
    return 0, 1

# ============================================================
# Execute Weather ETL
# ============================================================
print('=== Weather ETL ===')
conn = get_conn()
imp, skip = etl_weather(conn)
conn.close()
print(f'\nResult: cached={imp}, failed={skip}')

# Verify
conn = get_conn()
with conn.cursor() as cur:
    cur.execute('SELECT context_type, request_key, payload_json, expires_at FROM external_context_cache')
    for row in cur.fetchall():
        payload = json.loads(row[2])
        risk = payload.get('risk_level', 'N/A')
        temp = payload.get('temperature_c', 'N/A')
        print(f'  {row[0]}: risk={risk}, temp={temp}C, expires={row[3]}')
conn.close()


---
## Part 14: Venue Language ETL — LASS Data Integration

**Data Source**: Language_Access_Secret_Shopper_(LASS)_Ratings_20260526.csv
**Target Table**: `venue_language`
**Coverage**: 442 Manhattan records, ~55 venues matched (GPS <100m)

**ETL Strategy**:
1. Read LASS data, filter Manhattan
2. Extract language tags from translated signs/documents
3. Match with venues using GPS coordinates
4. Determine language_support_level based on language count
5. Insert into venue_language table


In [ ]:
def parse_lass_languages(lang_str):
    """Parse language string into ISO codes."""
    if not lang_str or lang_str.strip() in ('None', 'N/A', '', 'One or more languages (specific language not recorded)'):
        return []
    lang_map = {
        'spanish': 'es', 'chinese': 'zh', 'russian': 'ru', 'korean': 'ko',
        'french': 'fr', 'haitian creole': 'ht', 'arabic': 'ar', 'bengali': 'bn',
        'polish': 'pl', 'italian': 'it', 'japanese': 'ja', 'vietnamese': 'vi',
        'yiddish': 'yi', 'hebrew': 'he', 'urdu': 'ur', 'gujarati': 'gu',
        'tagalog': 'tl', 'french-creole': 'ht', 'french creole': 'ht',
    }
    if 'designated citywide' in lang_str.lower() or 'at least' in lang_str.lower():
        return ['es', 'zh', 'ru', 'ko', 'fr', 'ht']
    parts = [p.strip().rstrip('.') for p in lang_str.split(',')]
    langs = []
    for part in parts:
        p = part.lower().strip()
        if p in lang_map:
            langs.append(lang_map[p])
        elif p.startswith('lang:'):
            langs.append(p.replace('lang:', ''))
    return list(set(langs))

def find_nearest_venue(cur, lat, lng, threshold=100):
    """Find nearest venue within threshold (meters) using Haversine in SQL."""
    cur.execute(
        "SELECT venue_id, (6371000 * ACOS("
        "  COS(RADIANS(%s)) * COS(RADIANS(latitude)) *"
        "  COS(RADIANS(longitude) - RADIANS(%s)) +"
        "  SIN(RADIANS(%s)) * SIN(RADIANS(latitude))"
        ")) AS dist "
        "FROM venues WHERE latitude != 0 AND longitude != 0 "
        "HAVING dist < %s ORDER BY dist LIMIT 1",
        (lat, lng, lat, threshold)
    )
    row = cur.fetchone()
    return row[0] if row else None

def etl_venue_language(conn, lass_data):
    """Import LASS language data into venue_language table.

    Strategy:
    - SQL Haversine for GPS matching (replaces O(n×m) Python loop)
    - Merge signs + docs language tags
    - One venue_id per record
    """
    imp, skip = 0, 0
    lass_manhattan = [r for r in lass_data if r.get('Borough', '').strip().lower() == 'manhattan']
    print(f'  LASS Manhattan records: {len(lass_manhattan)}')

    for lass in lass_manhattan:
        # Get GPS
        try:
            lat = float(lass.get('Latitude', '').strip())
            lng = float(lass.get('Longitude', '').strip())
        except (ValueError, TypeError) as error:
            log_etl_error('venue_language', lass.get('Facility Name', '<unknown>'), error)
            skip += 1
            continue
        if not (40.700 <= lat <= 40.880 and -74.020 <= lng <= -73.9):
            skip += 1; continue

        # Merge signs + docs languages
        signs_col = 'Languages in which the facility has translated signs relating to service being provided'
        docs_col = 'Languages in which the facility has translated documents'
        all_langs = list(set(
            parse_lass_languages(lass.get(signs_col, '')) +
            parse_lass_languages(lass.get(docs_col, ''))
        ))

        # Classify support level
        level = 'full' if len(all_langs) >= 3 else ('partial' if all_langs else 'none')

        # Find nearest venue before the write transaction.
        try:
            with conn.cursor() as cur:
                venue_id = find_nearest_venue(cur, lat, lng)
        except pymysql.MySQLError as error:
            log_etl_error('venue_language_match', lass.get('Facility Name', '<unknown>'), error)
            skip += 1
            continue
        if not venue_id:
            skip += 1
            continue
        statement = (
            'INSERT INTO venue_language (venue_id, language_tag, language_support_level, chatbot_enabled) '
            'VALUES (%s, %s, %s, FALSE) '
            'ON DUPLICATE KEY UPDATE language_tag = VALUES(language_tag), language_support_level = VALUES(language_support_level)',
            (venue_id, json.dumps(all_langs) if all_langs else None, level),
        )
        if etl_execute(conn, statement, source='venue_language', record_id=venue_id):
            imp += 1
        else:
            skip += 1

    print(f'  Inserted: {imp}, Skipped: {skip}')
    return imp, skip

# ============================================================
# Execute Venue Language ETL
# ============================================================
print('=== Venue Language ETL ===')
LASS_PATH = DATA_ROOT / 'Language_Access_Secret_Shopper_(LASS)_Ratings_20260526.csv'
with open(LASS_PATH, encoding='utf-8-sig') as f:
    lass_data = list(csv.DictReader(f))
print(f'Loaded LASS data: {len(lass_data)} records')

conn = get_conn()
imp, skip = etl_venue_language(conn, lass_data)
conn.close()
print(f'\nResult: inserted={imp}, skipped={skip}')


### Venue Language — Import Results

| Metric | Value |
|--------|-------|
| LASS Manhattan records | 442 |
| GPS matched venues | 63 |
| venue_language inserted | 63 |
| Skipped (no GPS match) | 379 |

**Skip Reasons**:
- LASS records are government service centers (HRA, DPR, DOHMH, etc.)
- Most don't match ClearPath venue types (healthcare, restroom, AED, ramps)
- Only 14.3% matched within 100m GPS threshold

---
## Part 15: Final Verification

In [ ]:
# Verify all tables and columns
conn = get_conn()
with conn.cursor() as cur:
    cur.execute('SHOW TABLES')
    tables = [r[0] for r in cur.fetchall()]
    print(f'Total tables: {len(tables)}')
    print('Tables:', sorted(tables))
    
    # Check venues columns
    cur.execute('DESCRIBE venues')
    venue_cols = [r[0] for r in cur.fetchall()]
    print(f'\nvenues columns ({len(venue_cols)}):', venue_cols)
    
    # Check new tables
    for table in ['venue_accessibility', 'venue_language', 'venue_warnings']:
        if table in tables:
            cur.execute(f'DESCRIBE {table}')
            cols = [r[0] for r in cur.fetchall()]
            print(f'\n{table} ({len(cols)}):', cols)
conn.close()

In [ ]:
# Sync schema file with actual database structure
# This ensures 001_clearpath_schema.sql reflects the current database

try:
    conn = get_conn()
    with conn.cursor() as cur:
        # Get CREATE TABLE statements from information_schema
        cur.execute("SHOW TABLES")
        tables = [r[0] for r in cur.fetchall()]
        
        schema_content = """CREATE DATABASE IF NOT EXISTS clearpath
  CHARACTER SET utf8mb4
  COLLATE utf8mb4_0900_ai_ci;

USE clearpath;

"""
        for table in sorted(tables):
            cur.execute(f"SHOW CREATE TABLE {table}")
            create_stmt = cur.fetchone()[1]
            schema_content += f"{create_stmt};\n\n"
        
        # Write to file
        with open(SCHEMA_PATH, 'w', encoding='utf-8') as f:
            f.write(schema_content)
        
        print(f'Schema file updated: {SCHEMA_PATH}')
        print(f'Tables synced: {len(tables)}')
    conn.close()
except Exception as e:
    print(f'Schema sync failed: {e}')

In [ ]:
# ============================================================
# Final Database Verification — Table Row Counts + District Distribution
# ============================================================
print("=" * 60)
print("DATABASE VERIFICATION: Table Row Counts")
print("=" * 60)

conn = get_conn()
with conn.cursor() as cur:
    cur.execute('SHOW TABLES')
    all_tables = [r[0] for r in cur.fetchall()]
    print(f'\nTotal tables: {len(all_tables)}')
    print(f'\n{"Table":<30} {"Rows":>10}')
    print('-' * 42)
    
    total_rows = 0
    for table in sorted(all_tables):
        cur.execute(f'SELECT COUNT(*) FROM {table}')
        count = cur.fetchone()[0]
        total_rows += count
        print(f'{table:<30} {count:>10}')
    
    print('-' * 42)
    print(f'{"TOTAL":<30} {total_rows:>10}')
    
    # Check venue_source_links = venues (1:1 relationship)
    cur.execute('SELECT COUNT(*) FROM venues')
    venues_count = cur.fetchone()[0]
    cur.execute('SELECT COUNT(*) FROM venue_source_links')
    vsl_count = cur.fetchone()[0]
    match = "✅ OK" if venues_count == vsl_count else "⚠️ MISMATCH"
    print(f'\nvenues vs venue_source_links: {venues_count} vs {vsl_count} {match}')
    
    # Check for duplicate venue_ids in venues
    cur.execute('SELECT venue_id, COUNT(*) as cnt FROM venues GROUP BY venue_id HAVING cnt > 1')
    dups = cur.fetchall()
    if dups:
        print(f'⚠️ Duplicate venue_ids found: {len(dups)}')
    else:
        print('✅ No duplicate venue_ids in venues')

    # District distribution
    print('\n' + '=' * 60)
    print("DISTRICT DISTRIBUTION")
    print('=' * 60)
    
    print('\n--- venues ---')
    cur.execute('SELECT district, COUNT(*) FROM venues GROUP BY district ORDER BY COUNT(*) DESC')
    for row in cur.fetchall():
        d = row[0] or 'NULL'
        print(f'  {d:<20} {row[1]:>8}')
    
    print('\n--- pedestrian_ramps ---')
    cur.execute('SELECT district, COUNT(*) FROM pedestrian_ramps GROUP BY district ORDER BY COUNT(*) DESC')
    for row in cur.fetchall():
        d = row[0] or 'NULL'
        print(f'  {d:<20} {row[1]:>8}')

conn.close()

---
## Part 16: Conclusion

### Data Source Volume

| Source | Filter | Expected | Actual | Status |
|--------|--------|----------|--------|--------|
| NYC Public Restrooms | Coords bbox | 358 | 349 | ✅ |
| Parks Toilets | Borough | 129 | 127 | ✅ |
| OSM Healthcare | Coords bbox | 900 | 900 | ✅ |
| NYS Health Facility | County | 454 | 431 | ✅ |
| AED Inventory | Borough | 3,393 | 3,279 | ✅ (deduped) |
| Pedestrian Ramps | Borough code | 23,625 | 23,625 | ✅ |
| Weather (NWS API) | API | — | 1 cached | ✅ |
| Venue Language (LASS) | GPS match | ~55 | 63 | ✅ |

### Database Summary

| Table | Rows | Description |
|-------|------|-------------|
| venues | 3,479 | Main venue table (with district zoning) |
| venue_source_links | 3,479 | Data source tracking |
| restroom_profiles | 476 | NYC + Parks restrooms |
| healthcare_profiles | 1,228 | NYS + OSM healthcare |
| emergency_assets | 3,279 | AED (deduped) |
| pedestrian_ramps | 23,625 | Manhattan ramps (with district zoning) |
| external_context_cache | 1 | Current weather + risk_level |
| venue_language | 63 | LASS language support data |
| user_reports | 0 | Runtime (empty) |
| report_confirmations | 0 | Runtime (empty) |
| busyness_scores | 0 | ML model (empty) |
| venue_accessibility | 0 | Pending (empty) |
| venue_warnings | 0 | Runtime (empty) |

### Key Findings

1. **AED Dedup**: 3,393 → 3,279 (3.4% duplicates removed by entity+address+floor)
2. **Healthcare Merge**: NYS 431 + OSM 797 = 1,228 (0 GPS matches at 30m threshold)
3. **Parks Toilets**: 127 records with placeholder coordinates (0,0) for future geocoding
4. **Weather ETL**: NWS API integrated, 1 cache record with risk_level (low/medium/high)
5. **Venue Language**: LASS data integrated, 63 venues with language support info
6. **District Zoning**: GPS→district mapping applied to venues + pedestrian_ramps (uptown/midtown_east/midtown_west/downtown)
7. **forecast_1h**: Changed from INT to JSON type, stores 12-hour prediction array `[h0..h11]`; forecast_4h/8h removed

### API Schema Updates

- ✅ venue_type enum: 8 values (restroom, healthcare, emergency_asset, clinic, pharmacy, hospital, dentist, laboratory)
- ✅ venues: 24 columns (added language, accessibility, weather, photos, rating, district)
- ✅ user_reports: 12 columns (added anonymous, description, photos, reported_by, expires_in_minutes)
- ✅ busyness_scores.forecast_1h: JSON type (12-hour prediction array)
- ✅ 3 new tables: venue_accessibility, venue_language, venue_warnings
- ✅ Schema file synced with database

### Pending Work

| Task | Priority | Description |
|------|----------|-------------|
| venue_accessibility fill | Medium | Extract from OSM wheelchair tags |
| venue_warnings fill | Low | Runtime dynamic updates |
| Geocoding Parks Toilets | Medium | 127 records need Nominatim geocoding |
| OSM name cleaning | Low | 18 records with no name field |